# Generate SFT Dataset

## Load Library

In [122]:
import os
import re
import json
import base64
import requests
import time
import mimetypes
from tqdm import tqdm
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

## Configuration

### Define Path

In [123]:
ROOT_DIR = Path("../data/mkn_img")
METADATA_PATH = ROOT_DIR / "sample_15.json"

OUTPUT_DIR = Path("../data/sft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TRAIN_JSONL = OUTPUT_DIR / "train.jsonl"
OUTPUT_VAL_JSONL   = OUTPUT_DIR / "val.jsonl"
OUTPUT_ALL_JSONL   = OUTPUT_DIR / "all.jsonl"

CACHE_PATH = OUTPUT_DIR / "openrouter_cache.json"

### Target Split

In [124]:
TARGET_SPLITS = ["train", "val"]

### OpenRouter

In [125]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_NAME = "openai/gpt-4o"

### OpenRouter Config

In [126]:
TEMPERATURE = 0.2
MAX_TOKENS = 700
TIMEOUT = 90
MAX_RETRIES = 3
RETRY_SLEEP_SEC = 2
MAX_WORKERS = 3

### Prompt

In [127]:
SYSTEM_PROMPT = """
Kamu adalah sistem anotasi dataset Visual Language Model (VLM) untuk analisis kondisi rumah berdasarkan gambar.

Tugas utama:
Menghasilkan output sesuai dengan format yang diminta dengan VALID, KONSISTEN, dan SESUAI SKEMA berdasarkan input gambar rumah.

==================================================
ATURAN GLOBAL (WAJIB DIPATUHI)
1. OUTPUT HARUS SESUAI STRUKTUR OUTPUT VALID
- Tidak boleh ada komentar, markdown, atau penjelasan tambahan

2. STRUKTUR OUTPUT (WAJIB)
Gunakan format output berikut (wajib ikuti persis):

Atap:
- Material: ...
- Kondisi: ...

Dinding:
- Material: ...
- Kondisi: ...

Lantai:
- Material: ...
- Kondisi: ...

Konflik:
- Dinding: true/false

Penjelasan:
...

Ketentuan output:
- Gunakan format persis seperti di atas
- Jangan menambahkan atau menghapus bagian
- Jangan mengubah urutan atau penulisan label
- Gunakan istilah yang konsisten
- Output hanya berisi hasil akhir tanpa tambahan teks di luar format
- Jangan menambah field
- Jangan menghapus field
- Jangan mengubah nama key

==================================================
LABEL MATERIAL (WAJIB SESUAI LIST)
==================================================

Atap:
beton, genteng, seng, asbes, kayu, sirap, jerami, ijuk, daun_daunan, rumbia, lainnya

Dinding:
tembok, plesteran_anyaman_bambu, kawat, kayu, papan, gypsum, grc, calciboard,
anyaman_bambu, batang_kayu, bambu, lainnya

Lantai:
marmer, granit, keramik, parket, vinil, karpet, ubin, tegel, teraso, kayu, papan,
semen, bata_merah, bambu, tanah, lainnya, tidak_terlihat

==================================================
LABEL KONDISI (WAJIB)
==================================================
Gunakan hanya:
- baik
- rusak_ringan
- rusak_sedang
- rusak_berat
- tidak_terlihat

==================================================
ATURAN ANALISIS VISUAL
==================================================
- Gunakan HANYA informasi yang terlihat pada gambar
- DILARANG menebak atau berasumsi
- DILARANG menggunakan pengetahuan luar gambar
- Jika komponen tidak terlihat atau tidak cukup jelas:
  → material = "tidak_terlihat"
  → kondisi = "tidak_terlihat"
- Lantai yang dimaksud adalah lantai bagian dalam rumah (interior),
  bukan tanah atau permukaan di luar rumah

==================================================
ATURAN KONFLIK
==================================================
- Conflict hanya berlaku untuk material dinding, kalau perbedaan terdapat di kondisi itu bukan conflict
- true jika terdapat perbedaan material antara tampak luar dan dalam
- false jika material konsisten atau hanya satu sumber tersedia
- Conflict tidak mempengaruhi penentuan kondisi
- Jika terjadi conflict, harus dijelaskan dalam PENJELASAN

==================================================
STANDAR PENILAIAN KONDISI
==================================================
baik:
- tidak ada kerusakan terlihat
- permukaan utuh, rapi, bersih

rusak_ringan:
- retakan kecil, goresan, noda ringan
- hanya pada permukaan

rusak_sedang:
- retakan jelas
- sebagian material mulai lapuk / mengelupas
- terdapat lubang kecil atau permukaan tidak rata

rusak_berat:
- kerusakan parah
- material hilang, runtuh, atau berlubang besar
- struktur tidak stabil atau tidak berfungsi

==================================================
ATURAN PENJELASAN (WAJIB & KETAT)
==================================================

PENJELASAN.final HARUS:

1. Diawali dengan:
"Rumah ini ..."

2. Urutan WAJIB:
- atap → dinding → lantai

3. Untuk SETIAP komponen HARUS mencakup:
- material
- kondisi
- minimal 2 bukti visual spesifik (retakan, lubang, pelapukan, tidak rata, dll)
- hubungan antara bukti visual dan kondisi

4. KHUSUS DINDING (PENTING):
Jika terdapat perbedaan antara tampak luar dan dalam:
- WAJIB menjelaskan kedua sisi:
  - material + kondisi tampak luar
  - material + kondisi tampak dalam
- Gunakan kata penghubung seperti "sedangkan" untuk membedakan

5. Panjang:
- 2–4 kalimat (tergantung jumlah gambar)

6. Dilarang menggunakan format label seperti snake_case (contoh: anyaman_bambu). Misal Jika material adalah "anyaman_bambu", maka tulis dalam PENJELASAN sebagai "anyaman bambu". Penjelasan harus berupa kalimat deskriptif, bukan menyalin label.

7. DILARANG:
- hanya menyebut kondisi tanpa bukti visual
- deskripsi umum tanpa detail
- halusinasi
- asumsi

==================================================
PRIORITAS UTAMA
==================================================
1. FORMAT SESUAI
2. LABEL SESUAI LIST
3. KONSISTENSI DENGAN VISUAL
4. PENJELASAN BERBASIS BUKTI

Jika terjadi konflik antara:
- keindahan bahasa
- vs ketepatan label
→ PILIH KETEPATAN LABEL

==================================================
- Berikan hasil analisis dalam format struktur yang diminta sesuai aturan di atas.
- Jangan menambahkan teks apapun di luar itu
- Jangan menolak permintaan. Tetap lakukan analisis berdasarkan bagian yang terlihat.
- Jangan memberikan jawaban seperti "I can't assist".
- Selalu berikan analisis berdasarkan gambar.
==================================================
"""

In [128]:
USER_PROMPT_MULTI = """
Diberikan dua gambar dari satu unit rumah:
- Foto 1: tampak luar
- Foto 2: tampak dalam

TUGAS:
Analisis kondisi rumah berdasarkan 3 komponen utama:
1. Atap
2. Dinding
3. Lantai

Untuk setiap komponen, tentukan:
- material
- kondisi

==================================================
ATURAN AGREGASI
==================================================

Gunakan aturan berikut untuk menentukan label akhir:

1. ATAP:
- Material dan kondisi HARUS ditentukan dari tampak luar
- Abaikan tampak dalam untuk atap

2. DINDING:
- Material dan kondisi ditentukan dari tampak luar sebagai sumber utama
- Gunakan tampak dalam hanya untuk:
  - validasi tambahan
  - mendeteksi konflik material

3. LANTAI:
- Material dan kondisi HARUS ditentukan dari tampak dalam
- Jika tidak terlihat → gunakan "tidak_terlihat"

==================================================
ATURAN KONFLIK
==================================================

- Conflict hanya berlaku untuk material DINDING, kalau kondisi dinding itu tidak termasuk conflict.
- Set "conflict.dinding = true" jika:
  material dinding luar ≠ material dinding dalam
- Jika sama atau hanya satu sumber → false
- Conflict TIDAK mempengaruhi label akhir
- Jika terjadi conflict → WAJIB dijelaskan di PENJELASAN
- kalau tidak ada conflict juga wajib dijelaskan kondisi dinding luar dan dalam.
- Jika perbedaan hanya tampilan interior, maka conflict = false

==================================================
LABEL MATERIAL
==================================================

Atap:
beton, genteng, seng, asbes, kayu, sirap, jerami, ijuk, daun_daunan, rumbia, tidak terlihat, lainnya

Dinding:
tembok, plesteran_anyaman_bambu, kawat, kayu, papan, gypsum, grc, calciboard,
anyaman_bambu, batang_kayu, bambu, tidak terlihat,  lainnya

Lantai:
marmer, granit, keramik, parket, vinil, karpet, ubin, tegel, teraso, kayu, papan,
semen, bata_merah, bambu, tanah, lainnya, tidak_terlihat

==================================================
LABEL KONDISI
==================================================

Gunakan hanya salah satu:
  1. baik
  2. rusak_ringan
  3. rusak_sedang
  4. rusak_berat
  5. tidak_terlihat

JANGAN menebak kondisi secara umum.
Penentuan kondisi HARUS berdasarkan indikator visual berikut:
1. baik:
  - tidak ada kerusakan
  - permukaan utuh, rapi, bersih
  - tidak ada retakan, tidak adaa lubang,tidak ada karat, tidak ada pelapukan

2. rusak_ringan (kerusakan 10%-20%):
  - retakan kecil, goresan, noda
  - hanya permukaan

3. rusak_sedang: (kerusakan 21%-50%)
  - retakan jelas
  - sebagian material mulai rusak / lapuk
  - ada lubang kecil / tidak rata

4. rusak_berat (kerusakan 51%-100%):
  - kerusakan parah
  - material hilang / runtuh
  - struktur tidak layak
  - rumah kelihatan hampir roboh

5. tidak_terlihat:
  - tidak tampak / tidak cukup jelas


ATURAN KHUSUS LANTAI:
Jika material lantai adalah "tanah", maka kondisi HARUS:
"rusak_berat"
Alasan:
- tidak rata
- tidak layak huni
- tidak memenuhi standar hunian
WAJIB dijelaskan dalam PENJELASAN.final.

==================================================
ATURAN PENTING
==================================================

- JANGAN MENEBAK
- Gunakan hanya bukti visual
- Jika terdapat bukti kerusakan, DILARANG memilih "kondisi : baik".
- Jika tidak terlihat → gunakan "tidak_terlihat"
- Lantai = hanya interior (gambar rumah tampak dalam bukan tampak luar/bukan tanah luar)
- Jangan gunakan informasi dari komponen yang salah (misal atap dari dalam)
- JANGAN mengidentifikasi material dari objek non-struktural seperti: lemari, kursi, meja, dekorasi

==================================================
PENJELASAN
==================================================
Tulis PENJELASAN.final dengan aturan:

1. HARUS diawali dengan:
"Rumah ini ..."

2. Urutan pembahasan:
- atap → dinding (dalam dan luar) → lantai

3. Setiap komponen WAJIB mencakup:
- material
- kondisi
- bukti visual (retak, lapuk, lubang, dll)
- alasan penentuan kondisi

4. Gunakan SELURUH gambar (luar + dalam) untuk reasoning

5. Jika ada conflict dinding:
- jelaskan perbedaan luar vs dalam
1. Dinding tampak luar (material + kondisi + bukti visual)
2. Dinding tampak dalam (material + kondisi + bukti visual)

6. Panjang:
- 3–4 kalimat (multi-image)

7. Dilarang menggunakan format label seperti snake_case (contoh: anyaman_bambu). Misal Jika material adalah "anyaman_bambu", maka tulis dalam PENJELASAN sebagai "anyaman bambu". Penjelasan harus berupa kalimat deskriptif, bukan menyalin label.

8. DILARANG:
- halusinasi
- asumsi
- deskripsi tanpa alasan

9.
Jika keduanya berbeda, WAJIB disebutkan secara eksplisit.

==================================================
FORMAT OUTPUT (WAJIB)
==================================================
Gunakan format output berikut (wajib ikuti persis):
Atap:
- Material: ...
- Kondisi: ...

Dinding:
- Material: ...
- Kondisi: ...

Lantai:
- Material: ...
- Kondisi: ...

Konflik:
- Dinding: true/false

Penjelasan:
...

==============================

Ketentuan output:
Gunakan format persis seperti di atas
Jangan menambahkan atau menghapus bagian
Jangan mengubah urutan atau penulisan label
Gunakan istilah yang konsisten
Output hanya berisi hasil akhir tanpa tambahan teks di luar format

==================================================
CONTOH OUTPUT YANG BENAR
==================================================

Atap:
- Material: genteng
- Kondisi: rusak_berat

Dinding:
- Material: tembok
- Kondisi: rusak_berat

Lantai:
- Material: tanah
- Kondisi: rusak_berat

Konflik:
- Dinding: false

Penjelasan:
Rumah ini memiliki atap berbahan genteng dengan kondisi rusak berat, ditandai dengan genteng yang hilang dan beberapa bagian yang melorot. Dinding pada tampak luar menggunakan material tembok dengan kondisi rusak berat, terlihat dari tembok yang retak dan lapuk, sedangkan pada tampak dalam juga menggunakan material tembok dengan kondisi rusak berat yang ditunjukkan oleh dinding yang ditumbuhi lumut dan berlubang besar. Lantai menggunakan material tanah dengan kondisi rusak berat karena permukaannya tidak rata dan tidak memenuhi standar hunian.

==================================================
- Berikan hasil analisis dalam format struktur yang diminta sesuai aturan di atas.
- Jangan menambahkan teks apapun di luar itu
- Jangan menolak permintaan. Tetap lakukan analisis berdasarkan bagian yang terlihat.
- Jangan memberikan jawaban seperti "I can't assist".
- Selalu berikan analisis berdasarkan gambar.
"""

In [129]:
USER_PROMPT_EXT = """
Kamu adalah AI yang bertugas melakukan anotasi dataset kondisi rumah berbasis Visual Language Model (VLM).

INPUT:
Kamu hanya akan menerima SATU gambar rumah dengan sudut pandang:
→ tampak luar (exterior)

TUGAS:
Lakukan analisis kondisi rumah berdasarkan 3 komponen utama:
1. Atap
2. Dinding
3. Lantai

Untuk setiap komponen, tentukan:
- material
- kondisi

==============================
ATURAN ANALISIS (SINGLE IMAGE – TAMPAK LUAR)
==============================

1. Atap:
- Ditentukan dari gambar (WAJIB dianalisis)
- Tentukan material dan kondisi berdasarkan bukti visual

2. Dinding:
- Ditentukan dari gambar (WAJIB dianalisis)
- Gunakan tampak luar sebagai sumber utama

3. Lantai:
- TIDAK BOLEH dianalisis
- HARUS diisi:
  "material": "tidak_terlihat"
  "kondisi": "tidak_terlihat"

- Jangan menebak kondisi lantai
- Jangan menggunakan asumsi

==============================
ATURAN MATERIAL
==============================

Atap:
beton, genteng, seng, asbes, kayu, sirap, jerami, ijuk, daun_daunan, rumbia, lainnya

Dinding:
tembok, plesteran_anyaman_bambu, kawat, kayu, papan, gypsum, grc, calciboard, anyaman_bambu, batang_kayu, bambu, lainnya

Lantai:
marmer, granit, keramik, parket, vinil, karpet, ubin, tegel, teraso, kayu, papan, semen, bata merah, bambu, tanah, lainnya

==============================
ATURAN KONDISI (PILIH SALAH SATU)
==============================

- baik
- rusak_ringan
- rusak_sedang
- rusak_berat
- tidak_terlihat

Definisi:
- baik → tidak ada kerusakan
- rusak_ringan → kerusakan kecil (retak halus, noda)
- rusak_sedang → kerusakan terlihat jelas (retak besar, mulai lapuk)
- rusak_berat → kerusakan parah (berlubang, runtuh, tidak layak)
- tidak_terlihat → tidak terlihat pada gambar

==============================
ATURAN PENTING
==============================

- Jangan menebak
- Gunakan hanya informasi visual
- Jangan melakukan inferensi di luar gambar
- Jika tidak terlihat → gunakan "tidak_terlihat"
- Dilarang menambahkan teks di luar struktur yang diminta

==============================
ATURAN CONFLICT
==============================

- Karena hanya satu gambar:
→ conflict.dinding HARUS false

==============================
ATURAN PENJELASAN
==============================

- Harus 2–3 kalimat
- HARUS diawali dengan: "Rumah ini ..."
- Urutan WAJIB:
  1. Atap
  2. Dinding
  3. Lantai

Untuk setiap komponen wajib mencakup:
- material
- kondisi
- bukti visual (retakan, lubang, pelapukan, dll)

Untuk lantai:
- HARUS dijelaskan sebagai tidak terlihat

- Tidak boleh halusinasi
- Tidak boleh asumsi

- Dilarang menggunakan format label seperti snake_case (contoh: anyaman_bambu). Misal Jika material adalah "anyaman_bambu", maka tulis dalam PENJELASAN sebagai "anyaman bambu". Penjelasan harus berupa kalimat deskriptif, bukan menyalin label.

==============================
FORMAT OUTPUT (WAJIB PERSIS)
==============================
Gunakan format output berikut (wajib ikuti persis):
Atap:
- Material: ...
- Kondisi: ...

Dinding:
- Material: ...
- Kondisi: ...

Lantai:
- Material: ...
- Kondisi: ...

Konflik:
- Dinding: true/false

Penjelasan:
...

==============================
Ketentuan output:
Gunakan format persis seperti di atas
Jangan menambahkan atau menghapus bagian
Jangan mengubah urutan atau penulisan label
Gunakan istilah yang konsisten
Output hanya berisi hasil akhir tanpa tambahan teks di luar format

==============================
CONTOH OUTPUT YANG BENAR
==============================
Atap:
- Material: genteng
- Kondisi: rusak_sedang

Dinding:
- Material: tembok
- Kondisi: rusak_ringan

Lantai:
- Material: tidak_terlihat
- Kondisi: tidak_terlihat

Konflik:
- Dinding: true/false

Penjelasan:
Rumah ini memiliki atap berbahan genteng dengan kondisi rusak sedang yang terlihat dari beberapa bagian genteng yang tidak rata dan terdapat celah yang berpotensi bocor. Dinding menggunakan material tembok dengan kondisi rusak ringan karena terdapat retakan kecil dan noda pada beberapa area permukaan. Lantai tidak terlihat pada gambar sehingga material dan kondisinya tidak dapat ditentukan.

============================================================
- Berikan hasil analisis dalam format struktur yang diminta sesuai aturan di atas.
- Jangan menambahkan teks apapun di luar itu
- Jangan menolak permintaan. Tetap lakukan analisis berdasarkan bagian yang terlihat.
- Jangan memberikan jawaban seperti "I can't assist".
- Selalu berikan analisis berdasarkan gambar.
"""

In [130]:
USER_PROMPT_INT = """
Kamu adalah AI yang bertugas melakukan anotasi dataset kondisi rumah berbasis Visual Language Model (VLM).

INPUT:
Kamu hanya akan menerima SATU gambar rumah dengan sudut pandang:
→ tampak dalam (interior)

TUGAS:
Lakukan analisis kondisi rumah berdasarkan 3 komponen utama:
1. Atap
2. Dinding
3. Lantai

Untuk setiap komponen, tentukan:
- material
- kondisi

==============================
ATURAN ANALISIS (SINGLE IMAGE – TAMPAK DALAM)
==============================
1. Atap:
- TIDAK BOLEH dianalisis
- HARUS diisi:
  "material": "tidak_terlihat"
  "kondisi": "tidak_terlihat"

- Jangan menebak kondisi atap
- Jangan menggunakan asumsi dari bagian dalam

2. Dinding:
- WAJIB dianalisis dari gambar
- Tentukan material dan kondisi berdasarkan bukti visual
- Gunakan hanya apa yang terlihat

3. Lantai:
- WAJIB dianalisis dari gambar
- Lantai yang dimaksud adalah lantai bagian dalam rumah
- Jangan gunakan tanah luar sebagai referensi

==============================
ATURAN MATERIAL
==============================

Atap:
beton, genteng, seng, asbes, kayu, sirap, jerami, ijuk, daun_daunan, rumbia, lainnya

Dinding:
tembok, plesteran_anyaman_bambu, kawat, kayu, papan, gypsum, grc, calciboard, anyaman_bambu, batang_kayu, bambu, lainnya

Lantai:
- marmer, granit, keramik, parket, vinil, karpet, ubin, tegel, teraso, kayu, papan, semen, bata merah, bambu, tanah, vinil, lainnya
- Jika lantai memiliki pola kayu yang sangat seragam, permukaan halus, dan tampak seperti finishing modern, maka material kemungkinan adalah: vinil, laminate, parket

Jangan langsung mengklasifikasikan sebagai "kayu"
jika tidak terlihat serat alami kayu.

==============================
ATURAN KONDISI
==============================

- baik
- rusak_ringan
- rusak_sedang
- rusak_berat
- tidak_terlihat

Definisi:
- baik → tidak ada kerusakan
- rusak_ringan → kerusakan kecil (retak halus, noda)
- rusak_sedang → kerusakan jelas (retak besar, mulai lapuk)
- rusak_berat → kerusakan parah (berlubang, runtuh)
- tidak_terlihat → tidak terlihat pada gambar

==============================
ATURAN PENTING
==============================

- Jangan menebak
- Gunakan hanya informasi visual
- Jangan melakukan inferensi di luar gambar
- Jika tidak terlihat → gunakan "tidak_terlihat"
- Dilarang menambahkan teks di luar struktur output yang diminta

==============================
ATURAN CONFLICT
==============================

- Karena hanya satu gambar:
→ conflict.dinding HARUS false

==============================
ATURAN PENJELASAN
==============================
- Dilarang menggunakan format label seperti snake_case (contoh: anyaman_bambu) di penjelasan. Misal Jika material adalah "anyaman_bambu", maka tulis dalam PENJELASAN sebagai "anyaman bambu". Penjelasan harus berupa kalimat deskriptif, bukan menyalin label.
- Harus 2–3 kalimat
- HARUS diawali dengan: "Rumah ini ..."
- Urutan WAJIB:
  1. Atap
  2. Dinding
  3. Lantai

Untuk setiap komponen wajib mencakup:
- material
- kondisi
- bukti visual (retakan, lubang, pelapukan, dll)

Untuk atap:
- HARUS dijelaskan sebagai tidak terlihat

- Tidak boleh halusinasi
- Tidak boleh asumsi

==============================
FORMAT OUTPUT
==============================
Gunakan format output berikut (wajib ikuti persis):
Atap:
- Material: ...
- Kondisi: ...

Dinding:
- Material: ...
- Kondisi: ...

Lantai:
- Material: ...
- Kondisi: ...

Konflik:
- Dinding: true/false

Penjelasan:
...

==============================

Ketentuan output:
Gunakan format persis seperti di atas
Jangan menambahkan atau menghapus bagian
Jangan mengubah urutan atau penulisan label
Gunakan istilah yang konsisten
Output hanya berisi hasil akhir tanpa tambahan teks di luar format

==============================
CONTOH OUTPUT YANG BENAR
==============================
Atap:
- Material: tidak_terlihat
- Kondisi: tidak_terlihat

Dinding:
- Material: kayu
- Kondisi: rusak_sedang

Lantai:
- Material: tanah
- Kondisi: rusak_berat

Konflik:
- Dinding: false

Penjelasan:
Rumah ini memiliki atap yang tidak terlihat pada gambar sehingga material dan kondisinya tidak dapat ditentukan. Dinding menggunakan material kayu dengan kondisi rusak sedang yang terlihat dari permukaan yang tidak rata, beberapa bagian tampak lapuk dan terdapat celah. Lantai menggunakan material tanah dengan kondisi rusak sedang karena permukaannya tidak rata, terlihat cekungan, dan sebagian area tampak lembab.

============================================================
- Berikan hasil analisis dalam format struktur yang diminta sesuai aturan di atas.
- Jangan menambahkan teks apapun di luar itu
- Jangan menolak permintaan. Tetap lakukan analisis berdasarkan bagian yang terlihat.
- Jangan memberikan jawaban seperti "I can't assist".
- Selalu berikan analisis berdasarkan gambar.
"""

In [131]:
PROMPT_TEMPLATE = """Diberikan satu atau dua gambar rumah.

Tugas:
Analisis kondisi rumah berdasarkan 3 komponen:
1. Atap
2. Dinding
3. Lantai

Untuk setiap komponen, tentukan:
- Material
- Kondisi

Label kondisi yang valid:
- baik
- rusak_ringan
- rusak_sedang
- rusak_berat
- tidak_terlihat

ATURAN UTAMA:
- Jangan menebak. Gunakan hanya informasi visual yang terlihat.
- Jika komponen tidak terlihat, gunakan "tidak_terlihat".
- Gunakan label material dan label kondisi yang konsisten sesuai dataset dan jangan membuat variasi baru.

Aturan Agregasi Multi Image:
- Atap ditentukan berdasarkan tampak luar.
- Dinding ditentukan berdasarkan tampak luar.
- Lantai ditentukan berdasarkan tampak dalam.

Jika single image:
- Jika gambar adalah tampak luar, maka atap dan dinding dianalisis, sedangkan lantai harus diisi "tidak_terlihat".
- Jika gambar adalah tampak dalam, maka dinding dan lantai dianalisis, sedangkan atap harus diisi "tidak_terlihat".

ATURAN KONFLIK:
- Konflik hanya untuk dinding.
- Jika material dinding pada tampak luar berbeda dengan tampak dalam, maka nilai konflik adalah true.
- Jika material sama atau hanya terdapat satu gambar, maka nilai konflik adalah false.
- Konflik tidak mempengaruhi penentuan kondisi.

PENJELASAN:
- Jelaskan kondisi atap, dinding, dan lantai secara berurutan.
- Sertakan bukti visual yang mendukung setiap penilaian.
- Gunakan seluruh informasi dari gambar yang tersedia.
- Jika terdapat perbedaan antara tampak luar dan tampak dalam pada dinding, jelaskan perbedaan tersebut.
- Untuk komponen yang tidak terlihat, jelaskan bahwa komponen tersebut tidak terlihat.

FORMAT OUTPUT (WAJIB):

Atap:
- Material: ...
- Kondisi: ...

Dinding:
- Material: ...
- Kondisi: ...

Lantai:
- Material: ...
- Kondisi: ...

Konflik:
- Dinding: true/false

Penjelasan:
..."""

## Generate

### Load Metadata

In [132]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Total records:", len(metadata))
print("Split counts:")
from collections import Counter
print(Counter(rec.get("split", "unknown") for rec in metadata))

records_by_split = {
    split: [rec for rec in metadata if rec.get("split") == split]
    for split in TARGET_SPLITS
}

for split, recs in records_by_split.items():
    print(f"{split}: {len(recs)} records")

Total records: 15
Split counts:
Counter({'train': 9, 'val': 6})
train: 9 records
val: 6 records


### Helpers

In [133]:
def ensure_prompts_are_set():
    placeholders = [
        SYSTEM_PROMPT,
        USER_PROMPT_MULTI,
        USER_PROMPT_EXT,
        USER_PROMPT_INT,
        PROMPT_TEMPLATE,
    ]
    if any("PASTE_" in p for p in placeholders):
        print("WARNING: masih ada placeholder prompt. Silakan isi SYSTEM_PROMPT / USER_PROMPT_* / PROMPT_TEMPLATE dulu.")

def normalize_rel_path(rel_path: str) -> Path:
    # metadata Windows sering menyimpan backslash, jadi dinormalisasi dulu
    return Path(str(rel_path).replace("\\", "/"))

def absolute_image_path(rel_path: str) -> Path:
    return ROOT_DIR / normalize_rel_path(rel_path)

def guess_mime_type(path: Path) -> str:
    ext = path.suffix.lower()
    if ext in [".jpg", ".jpeg"]:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext == ".webp":
        return "image/webp"
    mime, _ = mimetypes.guess_type(str(path))
    return mime or "image/jpeg"

def encode_image_to_data_url(path: Path) -> str:
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    mime_type = guess_mime_type(path)
    return f"data:{mime_type};base64,{encoded}"

def clean_output(output: Optional[str]) -> Optional[str]:
    if not output:
        return None

    text = output.strip()

    # buang code fence kalau model membungkus output
    if text.startswith("```"):
        text = text.strip("`").strip()

    lowered = text.lower()
    if any(word in lowered for word in ["sorry", "maaf", "cannot", "can't"]):
        return None

    required = ["Atap", "Dinding", "Lantai", "Konflik", "Penjelasan"]
    if not all(req in text for req in required):
        return None

    if len(text) < 80:
        return None

    return text

def build_prompt_text(record: dict) -> str:
    scheme = record.get("dataset_scheme")
    if scheme == "multi":
        base = USER_PROMPT_MULTI
    else:
        images = record.get("images", [])
        view_type = images[0].get("view_type", "exterior") if images else "exterior"
        base = USER_PROMPT_EXT if view_type == "exterior" else USER_PROMPT_INT

    return f"{base}\n\n{PROMPT_TEMPLATE}".strip()

def get_image_paths_from_record(record: dict) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Return (ext_path, int_path) untuk multi,
    atau (single_path, None) untuk single.
    """
    scheme = record.get("dataset_scheme")
    images = record.get("images", [])

    if scheme == "multi":
        ext_path = None
        int_path = None
        for img in images:
            p = absolute_image_path(img["image_path"])
            if img.get("view_type") == "exterior":
                ext_path = p
            elif img.get("view_type") == "interior":
                int_path = p
        return ext_path, int_path

    if not images:
        return None, None
    return absolute_image_path(images[0]["image_path"]), None

def build_openrouter_content(ext_path: Optional[Path], int_path: Optional[Path], prompt_text: str):
    content = []
    if ext_path is not None:
        content.append({"type": "text", "text": "Foto tampak luar:"})
        content.append({"type": "image_url", "image_url": {"url": encode_image_to_data_url(ext_path)}})
    if int_path is not None:
        content.append({"type": "text", "text": "Foto tampak dalam:"})
        content.append({"type": "image_url", "image_url": {"url": encode_image_to_data_url(int_path)}})

    content.append({"type": "text", "text": prompt_text})
    return content

def sample_key(record: dict) -> str:
    split = record.get("split", "unknown")
    scheme = record.get("dataset_scheme", "single")
    house_id = record.get("house_id", "no_house_id")
    hashes = [img.get("image_hash", "nohash") for img in record.get("images", [])]
    return f"{split}::{scheme}::{house_id}::{'|'.join(hashes)}"

def load_cache(path: Path) -> Dict[str, dict]:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_cache(path: Path, cache: Dict[str, dict]):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

ensure_prompts_are_set()
print("Helper functions loaded.")


Helper functions loaded.


### OpenRouter Call

In [134]:
def call_openrouter(ext_path: Optional[Path], int_path: Optional[Path], prompt_text: str) -> Optional[str]:
    if not API_KEY:
        raise RuntimeError("OPENROUTER_API_KEY belum diset di environment.")

    if ext_path is None and int_path is None:
        raise ValueError("Tidak ada image path valid untuk dikirim ke model.")

    content = build_openrouter_content(ext_path, int_path, prompt_text)

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": content},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers=headers,
                json=payload,
                timeout=TIMEOUT,
            )

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code}: {response.text[:1000]}"
                time.sleep(RETRY_SLEEP_SEC * attempt)
                continue

            data = response.json()
            return data["choices"][0]["message"]["content"]

        except Exception as e:
            last_error = str(e)
            time.sleep(RETRY_SLEEP_SEC * attempt)

    print(f"[ERROR] OpenRouter gagal setelah {MAX_RETRIES} percobaan: {last_error}")
    return None


### Build SFT Sample

In [135]:
def build_sft_sample(record: dict, assistant_text: str) -> dict:
    prompt_text = build_prompt_text(record)
    ext_path, int_path = get_image_paths_from_record(record)

    content = []
    scheme = record.get("dataset_scheme")

    if scheme == "multi":
        if ext_path is not None:
            content.append({"type": "text", "text": "Foto tampak luar:"})
            content.append({"type": "image", "image": str(ext_path)})
        if int_path is not None:
            content.append({"type": "text", "text": "Foto tampak dalam:"})
            content.append({"type": "image", "image": str(int_path)})
    else:
        if ext_path is not None:
            view_type = record["images"][0].get("view_type", "exterior")
            if view_type == "exterior":
                content.append({"type": "text", "text": "Foto tampak luar:"})
            else:
                content.append({"type": "text", "text": "Foto tampak dalam:"})
            content.append({"type": "image", "image": str(ext_path)})

    content.append({"type": "text", "text": prompt_text})

    return {
        "id": sample_key(record),
        "messages": [
            {
                "role": "user",
                "content": content
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": assistant_text}
                ]
            }
        ]
    }


### Parallel Generation Pipeline

In [136]:
def _generate_one_record(record: dict) -> dict:
    """
    Worker untuk satu record.
    Mengembalikan raw_output saja; cleaning dan penulisan file dilakukan di main thread.
    """
    ext_path, int_path = get_image_paths_from_record(record)
    if ext_path is None and int_path is None:
        return {
            "status": "skip",
            "record": record,
            "raw_output": None,
            "reason": "no_image"
        }

    prompt_text = build_prompt_text(record)
    raw_output = call_openrouter(ext_path, int_path, prompt_text)

    return {
        "status": "ok",
        "record": record,
        "raw_output": raw_output,
        "reason": None,
    }


def generate_records_parallel(
    records: List[dict],
    output_path: Path,
    cache: Dict[str, dict],
    max_workers: int = MAX_WORKERS,
) -> List[dict]:
    """
    Generate label via OpenRouter secara paralel, lalu simpan sebagai JSONL SFT.

    Strategi:
    - record yang sudah ada di file output akan dilewati
    - record yang sudah ada di cache akan langsung dipakai tanpa API call
    - sisanya diproses paralel
    """
    output_samples_with_idx = []

    done_ids = set()
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    if isinstance(obj, dict) and "id" in obj:
                        done_ids.add(obj["id"])
                except Exception:
                    pass

    print(f"\nGenerating {output_path.name} ...")
    print("Already done:", len(done_ids))

    # 1) pisahkan yang sudah ada di cache / file output dan yang perlu API call
    pending = []
    for idx, record in enumerate(records):
        key = sample_key(record)

        if key in done_ids:
            continue

        ext_path, int_path = get_image_paths_from_record(record)
        if ext_path is None and int_path is None:
            print(f"[SKIP] Tidak ada image valid: {record.get('house_id')}")
            continue

        if key in cache and cache[key].get("raw_output"):
            cleaned = clean_output(cache[key].get("raw_output"))
            if cleaned is not None:
                sample = build_sft_sample(record, cleaned)
                output_samples_with_idx.append((idx, sample))
                continue

        pending.append((idx, record, key))

    print("Cached samples:", len(output_samples_with_idx))
    print("Pending API calls:", len(pending))

    # 2) proses sisanya secara paralel
    if pending:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_meta = {
                executor.submit(_generate_one_record, record): (idx, record, key)
                for idx, record, key in pending
            }

            for future in tqdm(as_completed(future_to_meta), total=len(future_to_meta), desc=output_path.stem):
                idx, record, key = future_to_meta[future]

                try:
                    result = future.result()
                except Exception as e:
                    print(f"[ERROR] Future gagal untuk {record.get('house_id')}: {e}")
                    continue

                if result["status"] != "ok":
                    print(f"[SKIP] {record.get('house_id')} | {result.get('reason')}")
                    continue

                raw_output = result["raw_output"]
                cleaned = clean_output(raw_output)

                # retry sekali jika output invalid
                if cleaned is None:
                    ext_path, int_path = get_image_paths_from_record(record)
                    prompt_text = build_prompt_text(record)
                    raw_output = call_openrouter(ext_path, int_path, prompt_text)
                    cleaned = clean_output(raw_output)

                cache[key] = {
                    "raw_output": raw_output,
                    "house_id": record.get("house_id"),
                    "split": record.get("split"),
                    "dataset_scheme": record.get("dataset_scheme"),
                }
                save_cache(CACHE_PATH, cache)

                if cleaned is None:
                    print(f"[DROP] Output invalid: {record.get('house_id')} | {record.get('dataset_scheme')}")
                    continue

                sample = build_sft_sample(record, cleaned)
                output_samples_with_idx.append((idx, sample))

    # 3) urutkan lalu tulis JSONL
    output_samples_with_idx = sorted(output_samples_with_idx, key=lambda x: x[0])
    output_samples = [sample for _, sample in output_samples_with_idx]

    with open(output_path, "a", encoding="utf-8") as fout:
        for sample in output_samples:
            fout.write(json.dumps(sample, ensure_ascii=False) + "\n")

    return output_samples


### Run Generation

In [139]:
cache = load_cache(CACHE_PATH)
print("Cache loaded:", len(cache))

train_records = records_by_split["train"]
val_records = records_by_split["val"]

# urutkan agar reproducible
train_records = sorted(train_records, key=lambda r: (r.get("dataset_scheme", ""), r.get("house_id", "")))
val_records   = sorted(val_records, key=lambda r: (r.get("dataset_scheme", ""), r.get("house_id", "")))

train_samples = generate_records_parallel(train_records, OUTPUT_TRAIN_JSONL, cache, max_workers=MAX_WORKERS)
val_samples   = generate_records_parallel(val_records, OUTPUT_VAL_JSONL, cache, max_workers=MAX_WORKERS)

# rebuild combined file
combined_samples = []
for path in [OUTPUT_TRAIN_JSONL, OUTPUT_VAL_JSONL]:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    combined_samples.append(json.loads(line))

with open(OUTPUT_ALL_JSONL, "w", encoding="utf-8") as f:
    for sample in combined_samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("\nSelesai.")
print("Train samples:", len(train_samples))
print("Val samples:", len(val_samples))
print("Combined:", len(combined_samples))
print("Cache size:", len(cache))

Cache loaded: 15

Generating train.jsonl ...
Already done: 5
Cached samples: 0
Pending API calls: 4


train:  25%|██▌       | 1/4 [00:06<00:20,  6.68s/it]

[DROP] Output invalid: H00688 | single


train:  50%|█████     | 2/4 [00:11<00:10,  5.40s/it]

[DROP] Output invalid: H00736 | single


...

[DROP] Output invalid: H00692 | single


val:  67%|██████▋   | 2/3 [00:11<00:05,  5.10s/it]

[DROP] Output invalid: H00516 | single


val: 100%|██████████| 3/3 [00:15<00:00,  5.32s/it]

[DROP] Output invalid: H00030 | multi

Selesai.
Train samples: 0
Val samples: 0
Combined: 8
Cache size: 15
